# 漏洞代码匹配

为干净代码片段从漏洞知识库中检索语义最相似的含漏洞函数代码片段。

In [ ]:
import sys
import os
from pathlib import Path
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 向上两级到项目根目录
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


from src.utils.io import (
    load_json, load_jsonl, save_json, load_yaml
)
from src.utils.log import (
    setup_logging
)
from src.models.retriever.Base import create_embedder
from src.rag.vul.milvus_client import VulMilvusClient
from src.rag.vul.retriever import VulRetriever

setup_logging()

## 1. 加载配置

In [ ]:
db_config = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))
model_config = load_yaml(str(PROJECT_ROOT / db_config["model"]["retriever_config"]))

# 覆盖控制参数
FORCE_OVERWRITE = True # True：清除已有 generation_attack 字段并重新检索（覆盖模式）；False：跳过已有 generation_attack 的条目（断点续写模式）

# 将相对路径转为绝对路径
if not os.path.isabs(db_config["database"]["uri"]):
    db_config["database"]["uri"] = str(PROJECT_ROOT / db_config["database"]["uri"])
if not os.path.isabs(db_config["data"]["rag_base_file_path"]):
    db_config["data"]["rag_base_file_path"] = str(PROJECT_ROOT / db_config["data"]["rag_base_file_path"])
if not os.path.isabs(db_config["data"]["query_file_path"]):
    db_config["data"]["query_file_path"] = str(PROJECT_ROOT / db_config["data"]["query_file_path"])


print("=== Model Config ===")
for k, v in model_config.items():
    print(f"  {k}: {v}")

print("\n=== Database Config ===")
for k, v in db_config["database"].items():
    print(f"  {k}: {v}")


## 2. 加载检索器

In [ ]:
embedder = create_embedder(model_config)

# 快速验证
test_emb = embedder.embed_queries(["int main() { return 0; }"])
print(f"Test embedding shape: {test_emb.shape}")
print(f"Norm: {(test_emb[0] ** 2).sum() ** 0.5:.4f}")

## 3. 构建 Milvus 知识库
生成 dense embeddings 并连同原始文本一起插入 Milvus，BM25 sparse 向量由 Milvus 内置函数自动生成。

In [ ]:
milvus_client = VulMilvusClient(db_config)
collection_name = db_config["database"]["collection_name"]
already_built = (
    milvus_client.client.has_collection(collection_name)
    and milvus_client.get_collection_stats()["row_count"] > 0
)

if already_built:
    print("Knowledge base load successfully.")
else:
    # 加载数据
    kb_data = load_jsonl(db_config["data"]["rag_base_file_path"])
    print(f"Knowledge base: {len(kb_data)} entries")
    
    # 构建知识库
    milvus_client.build_knowledge_base(embedder, kb_data)
    print("Knowledge base built successfully.")


## 4. 加载投毒目标文件（poison_targets_set.json）

In [ ]:
from src.utils.io import load_json, save_json

# 将相对路径转为绝对路径
if not os.path.isabs(db_config["data"]["query_file_path"]):
    db_config["data"]["query_file_path"] = str(PROJECT_ROOT / db_config["data"]["query_file_path"])

query_file_path = db_config["data"]["query_file_path"]
poison_targets = load_json(query_file_path) # poison_targets_set.json 是 {CWE: [{entity, retrieval_attack}, ...]} 结构的 JSON dict

if FORCE_OVERWRITE:
    for cwe in poison_targets:
        for entry in poison_targets[cwe]:
            entry.pop("generation_attack", None)
    print("FORCE_OVERWRITE=True: 已清除所有已有 generation_attack 字段")

CWE_LIST = list(poison_targets.keys())
total_entities = sum(len(v) for v in poison_targets.values())
print(f"Loaded: {query_file_path}")
print(f"CWE groups: {CWE_LIST}")
print(f"Total entities: {total_entities}")
for cwe in CWE_LIST:
    entities = poison_targets[cwe]
    already_done = sum(1 for e in entities if "generation_attack" in e)
    print(f"  {cwe}: {len(entities)} entities ({already_done} already have generation_attack)")

## 5. 逐 CWE 执行相关漏洞检索

In [ ]:
retriever = VulRetriever(embedder, milvus_client)
top_k = db_config["retrieval"].get("top_k", 10)
retrieval_strategy = db_config["retrieval"].get("retrieval_strategy", "hybrid")
cluster_search = db_config["retrieval"].get("cluster_search", False)
alpha = db_config["retrieval"].get("alpha_weight", 0.85)

print(f"Retrieval params: top_k={top_k}, strategy={retrieval_strategy}, cluster_search={cluster_search}")

all_results = {}  # {cwe: [retrieval_result, ...]}，与 pending entities 对齐

for cwe in CWE_LIST:
    entities = poison_targets[cwe]

    print(f"\n[{cwe}] 开始检索，共 {len(entities)} 条")

    # 将 entity 字段展平为检索器所需的 flat list
    query_list = [
        {
            "id": e["entity"]["id"],
            "source": e["entity"]["source"],
            "buggy_code": e["entity"]["buggy_code"],
            "fixed_code": e["entity"]["fixed_code"],
        }
        for e in entities
    ]

    if retrieval_strategy == "dense":
        results = retriever.dense_retrieve(
            query_list, top_k=top_k, cluster_search=cluster_search,
            query_target="fixed_code", cwe=cwe,
        )
    elif retrieval_strategy == "sparse":
        results = retriever.sparse_retrieve(
            query_list, top_k=top_k, cluster_search=cluster_search,
            query_target="fixed_code", cwe=cwe,
        )
    else:  # hybrid
        results = retriever.hybrid_retrieve(
            query_list, top_k=top_k, alpha=alpha, cluster_search=cluster_search,
            query_target="fixed_code", cwe=cwe,
        )

    all_results[cwe] = results
    print(f"[{cwe}] 检索完成，共 {len(results)} 条结果")
    if results:
        sample_hits = results[0]["retrieval_results"]
        print(f"  样例: entity_id={results[0]['id']} -> {len(sample_hits)} hits")
        if sample_hits:
            h = sample_hits[0]
            print(f"  Top hit: doc_id={h['doc_id']}, cwe={h['cwe_id']}, score={h['score']:.4f}")

print("\n所有 CWE 检索完毕")

## 6. 将检索结果写回投毒目标文件（in-place 追加 generation_attack）

In [ ]:
merged_count = 0

for cwe in CWE_LIST:
    entities = poison_targets[cwe]
    cwe_results = all_results.get(cwe, [])

    # id -> retrieval_results 映射（本次新检索的条目）
    result_map = {str(r["id"]): r["retrieval_results"] for r in cwe_results}

    for entry in entities:
        entity_id = str(entry["entity"]["id"])
        hits = result_map.get(entity_id, [])
        entry["generation_attack"] = {
            "vinj_attack": {
                "relevant_vul": hits
            },
            "jailbreak_attack": {}
        }
        merged_count += 1

print(f"合并完成：新增 generation_attack 字段 {merged_count} 条")
for cwe in CWE_LIST:
    done = sum(1 for e in poison_targets[cwe] if "generation_attack" in e)
    total = len(poison_targets[cwe])
    print(f"  {cwe}: {done}/{total} 条已有 generation_attack")

## 7. 保存结果（覆盖写回 query_file）

In [ ]:
save_json(poison_targets, query_file_path)
print(f"检索结果已覆盖写回: {query_file_path}")

milvus_client.close()
print("Done!")

## 8. 结果分析
见src/preprocessing/vinj/retrieval_results_analysis.ipynb